# Add KaroSpace polygons back to AnnData

This notebook follows the current `karospace.integrate_polygon_annotations` implementation and shows a complete import workflow from exported KaroSpace polygon JSON into an `AnnData` object.

## What the current implementation does

- Loads annotation payload from a JSON file path or a Python mapping.
- Uses `polygon.cell_global_indices` first when present.
- Falls back to resolving `polygon.cell_local_indices` by section using payload `groupby` + `polygon.section_id`.
- Writes two `obs` columns:
  - `label_key` (default: `karospace_polygon_labels`) as pandas string dtype, `NA` for unlabeled cells.
  - `count_key` (default: `karospace_polygon_count`) as `int32`.
- Stores resolved polygon metadata in `adata.uns[uns_key]` (default: `karospace_polygon_annotations`).

In [1]:
from pathlib import Path
import json

import pandas as pd
import scanpy as sc
from karospace import integrate_polygon_annotations

# Update these paths for your dataset.
H5AD_PATH = Path("/Volumes/processing2/fetal_lung2/derived_scanpy/fetal_lung2_clustered.h5ad")
ANNOTATIONS_JSON = Path("/Users/chrislangseth/Downloads/karospace-annotations-2026-02-23T14-18-54-899Z.json")
OUTPUT_H5AD = H5AD_PATH.with_name(f"{H5AD_PATH.stem}_with_polygons.h5ad")

# Keep these aligned with integrate_polygon_annotations(...) call.
LABEL_KEY = "karospace_polygon_labels"
COUNT_KEY = "karospace_polygon_count"
UNS_KEY = "karospace_polygon_annotations"
DELIMITER = "|"

/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
# Load AnnData and inspect JSON metadata before integration.
adata = sc.read_h5ad(H5AD_PATH)
payload = json.loads(ANNOTATIONS_JSON.read_text(encoding="utf-8"))

print(f"Loaded adata with n_obs={adata.n_obs:,} and n_vars={adata.n_vars:,}")
print(f"Payload format: {payload.get('format')}")
print(f"Payload groupby: {payload.get('groupby')}")
print(f"Payload polygons: {len(payload.get('polygons', []))}")

if payload.get("groupby") and payload["groupby"] not in adata.obs.columns:
    print(
        f"Warning: groupby column '{payload['groupby']}' is not in adata.obs. "
        "Fallback from local indices to global indices will not be available."
    )

Loaded adata with n_obs=1,499,516 and n_vars=343
Payload format: karospace-polygon-annotations-v1
Payload groupby: sample_id
Payload polygons: 26


In [3]:
# Integrate polygon annotations using the current implementation defaults.
adata = integrate_polygon_annotations(
    adata,
    ANNOTATIONS_JSON,
    label_key=LABEL_KEY,
    count_key=COUNT_KEY,
    uns_key=UNS_KEY,
    delimiter=DELIMITER,
)

In [4]:
# Quick checks: how many cells got labels and what was stored.
n_annotated = int((adata.obs[COUNT_KEY] > 0).sum())
print(f"Cells with >=1 polygon label: {n_annotated:,}")
print(f"Polygons resolved: {adata.uns[UNS_KEY]['n_polygons']}")

adata.obs[[LABEL_KEY, COUNT_KEY]].head(10)

Cells with >=1 polygon label: 1,485,764
Polygons resolved: 26


,karospace_polygon_labels,karospace_polygon_count
obs_id,,
M-Goliath-A1_slide2:aaaabpbn-1,<NA>,0
M-Goliath-A1_slide2:aaaacjga-1,A1_slide2_s6,1
M-Goliath-A1_slide2:aaaaeekj-1,A1_slide2_s6,1
M-Goliath-A1_slide2:aaaafejn-1,A1_slide2_s6,1
M-Goliath-A1_slide2:aaaagced-1,A1_slide2_s6,1
M-Goliath-A1_slide2:aaaajbpo-1,A1_slide2_s6,1
M-Goliath-A1_slide2:aaabacjg-1,A1_slide2_s6,1
M-Goliath-A1_slide2:aaabebep-1,A1_slide2_s6,1
M-Goliath-A1_slide2:aaabfend-1,A1_slide2_s6,1


In [5]:
# Polygon-level summary from adata.uns[UNS_KEY]['polygons'].
poly_df = pd.DataFrame(adata.uns[UNS_KEY]["polygons"])
if not poly_df.empty:
    display(poly_df[["id", "label", "section_id", "n_cells"]].sort_values("id").head(20))
else:
    print("No polygons were resolved.")

,id,label,section_id,n_cells
0,1,A1_slide2_s1,M-Goliath-A1_slide2,63970
1,3,A1_slide2_s2,M-Goliath-A1_slide2,27383
2,4,A1_slide2_s3,M-Goliath-A1_slide2,56548
3,5,A1_slide2_s4,M-Goliath-A1_slide2,154388
4,7,A1_slide2_s5,M-Goliath-A1_slide2,70796
5,9,A1_slide2_s6,M-Goliath-A1_slide2,38513
6,11,A1_slide2_s7,M-Goliath-A1_slide2,36471
7,12,A1_slide2_s8,M-Goliath-A1_slide2,29059
8,13,A1_slide2_s9,M-Goliath-A1_slide2,15629
9,14,A1_slide2_s10,M-Goliath-A1_slide2,53787


In [8]:
adata.obs

,x_centroid,y_centroid,transcript_counts,control_probe_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,...,sample_id,cell_id,n_genes_by_counts,n_counts,n_genes,leiden_1.0,_scvi_batch,_scvi_labels,karospace_polygon_labels,karospace_polygon_count
obs_id,,,,,,,,,,,,,,,,,,,,,
M-Goliath-A1_slide2:aaaabpbn-1,691.379700,7592.583008,69,0,0,0,0,69.0,769.733465,7.134688,...,M-Goliath-A1_slide2,aaaabpbn-1,32,69.0,32,7,0,0,<NA>,0
M-Goliath-A1_slide2:aaaacjga-1,826.002136,7533.051270,110,0,0,0,0,110.0,74.688440,30.977189,...,M-Goliath-A1_slide2,aaaacjga-1,54,110.0,54,3,0,0,A1_slide2_s6,1
M-Goliath-A1_slide2:aaaaeekj-1,846.800659,7523.861328,71,0,0,0,0,71.0,51.342658,18.333438,...,M-Goliath-A1_slide2,aaaaeekj-1,36,71.0,36,1,0,0,A1_slide2_s6,1
M-Goliath-A1_slide2:aaaafejn-1,842.044739,7460.675781,28,0,0,0,0,28.0,296.992667,4.199531,...,M-Goliath-A1_slide2,aaaafejn-1,15,28.0,15,4,0,0,A1_slide2_s6,1
M-Goliath-A1_slide2:aaaagced-1,836.279663,7539.506836,47,0,0,0,0,47.0,32.693126,9.031250,...,M-Goliath-A1_slide2,aaaagced-1,28,47.0,28,0,0,0,A1_slide2_s6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M-Goliath-B1_slide1:oikpklop-1,10812.224609,11532.014648,143,0,0,0,0,143.0,75.456096,41.092189,...,M-Goliath-B1_slide1,oikpklop-1,50,143.0,50,2,1,0,A1_slide2_s17,1
M-Goliath-B1_slide1:oikplkmc-1,10853.613281,11532.392578,92,0,0,0,0,92.0,139.532818,18.243126,...,M-Goliath-B1_slide1,oikplkmc-1,40,92.0,40,2,1,0,A1_slide2_s17,1
M-Goliath-B1_slide1:oikpmlmo-1,10832.010742,11535.008789,74,0,0,0,0,74.0,72.972503,40.234220,...,M-Goliath-B1_slide1,oikpmlmo-1,36,74.0,36,2,1,0,A1_slide2_s17,1


In [15]:
adata.obs.karospace_polygon_labels = adata.obs.karospace_polygon_labels.str.split('|', expand = True)[0]

In [21]:
adata = adata[adata.obs.karospace_polygon_labels.notna()]

In [24]:
del adata.uns['karospace_polygon_annotations']['polygons']

In [25]:
# Save an updated h5ad with polygon labels/counts and full polygon metadata.
adata.write_h5ad(OUTPUT_H5AD)
print(f"Wrote: {OUTPUT_H5AD}")

Wrote: /Volumes/processing2/fetal_lung2/derived_scanpy/fetal_lung2_clustered_with_polygons.h5ad


## Optional: custom output keys

Use custom `obs`/`uns` keys if you want multiple annotation sets in the same file.

In [ ]:
# Example custom keys (run on a fresh adata load if you want both variants):
# adata_custom = sc.read_h5ad(H5AD_PATH)
# adata_custom = integrate_polygon_annotations(
#     adata_custom,
#     ANNOTATIONS_JSON,
#     label_key="lesion_labels",
#     count_key="lesion_label_count",
#     uns_key="lesion_polygons",
#     delimiter="|",
# )
# adata_custom.write_h5ad(H5AD_PATH.with_name(f"{H5AD_PATH.stem}_with_lesion_polygons.h5ad"))